In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=613362a1b44113547755a3caf065179b6d02eb513ca4d40a34bdbb7feb37fcd6
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [4]:
_backend = BasicSimulator()

# ── Quantum random bit generator ─────────────────────────────

def quantum_random_bits(n):
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)
    qc.measure(range(n), range(n))
    compiled = transpile(qc, _backend)
    counts = _backend.run(compiled, shots=1).result().get_counts()
    bitstring = list(counts.keys())[0].replace(' ', '')
    return [int(b) for b in reversed(bitstring)]

N = 20
DETECTION_THRESHOLD = 0.10

# ── ALICE ─────────────────────────────────────────────────────
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

# ── EVE ───────────────────────────────────────────────────────
# Eve intercepts every qubit, guessing a random basis each time.
# She measures, collapses the state, then re-encodes her result.
# When her basis differs from Alice's (~50% of the time), she
# forwards the wrong state — injecting errors Bob can detect.
eve_bases = quantum_random_bits(N)

def eve_intercept(bit, alice_basis, eve_basis):
    """Eve measures Alice's qubit in her own basis, returns her result."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if alice_basis == 1:
        qc.h(0)
    # Eve measures in her basis
    if eve_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, _backend)
    counts = _backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0])

# Eve intercepts all qubits and records her results
eve_results = [
    eve_intercept(alice_bits[i], alice_bases[i], eve_bases[i])
    for i in range(N)
]

# ── BOB ───────────────────────────────────────────────────────
# Bob receives Eve's re-encoded qubits (not Alice's originals).
bob_bases = quantum_random_bits(N)

def prepare_and_measure(bit, sender_basis, bob_basis):
    """Compose sender's encoding + Bob's measurement into one circuit."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if sender_basis == 1:
        qc.h(0)
    if bob_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, _backend)
    counts = _backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0])

# Bob measures Eve's re-encoded qubits
bob_results = [
    prepare_and_measure(eve_results[i], eve_bases[i], bob_bases[i])
    for i in range(N)
]

# ── Display ───────────────────────────────────────────────────

basis_label = lambda b: 's' if b == 0 else 'd'
qubit_label = lambda bit, basis: (str(bit) if basis == 0 else ('+' if bit == 0 else '-'))

print(f"{'Index:':<10}", ''.join(f"{i:<4}" for i in range(N)))
print(f"{'A bit:':<10}", ''.join(f"{alice_bits[i]:<4}"                              for i in range(N)))
print(f"{'A basis:':<10}", ''.join(f"{basis_label(alice_bases[i]):<4}"              for i in range(N)))
print(f"{'qubit:':<10}", ''.join(f"{qubit_label(alice_bits[i], alice_bases[i]):<4}" for i in range(N)))
print(f"{'E basis:':<10}", ''.join(f"{basis_label(eve_bases[i]):<4}"                for i in range(N)))
print(f"{'E bit:':<10}", ''.join(f"{eve_results[i]:<4}"                             for i in range(N)))
print(f"{'B basis:':<10}", ''.join(f"{basis_label(bob_bases[i]):<4}"                for i in range(N)))

bob_display = [
    str(bob_results[i]) if alice_bases[i] == bob_bases[i] else '?'
    for i in range(N)
]
print(f"{'B bit:':<10}", ''.join(f"{v:<4}" for v in bob_display))

# ── Sifting ───────────────────────────────────────────────────

matching  = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i]  for i in matching]
bob_key   = [bob_results[i] for i in matching]

print(f"\nMatching positions : {matching}")
print(f"Alice's sifted key : {alice_key}")
print(f"Bob's   sifted key : {bob_key}")

# ── Error check / attack detection ────────────────────────────

errors     = sum(a != b for a, b in zip(alice_key, bob_key))
error_rate = errors / len(alice_key) if alice_key else 0

print(f"\nErrors     : {errors}/{len(alice_key)} = {error_rate:.1%}")
print(f"Threshold  : {DETECTION_THRESHOLD:.0%}")

if error_rate > DETECTION_THRESHOLD:
    print("⚠ ATTACK DETECTED — aborting key exchange!")
else:
    print("✓ Error rate within threshold — Eve was lucky this run.")

Index:     0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  
A bit:     0   1   1   1   0   0   1   0   1   1   1   1   1   0   1   0   0   1   0   1   
A basis:   d   s   d   s   s   s   d   d   s   d   s   d   s   s   d   d   s   s   d   s   
qubit:     +   1   -   1   0   0   -   +   1   -   1   -   1   0   -   +   0   1   +   1   
E basis:   d   s   d   d   d   d   s   d   s   s   d   s   d   d   s   d   d   d   s   d   
E bit:     0   1   1   1   0   0   1   0   1   1   0   0   1   1   0   0   0   1   0   1   
B basis:   d   s   d   d   s   s   d   s   d   s   s   d   d   s   d   d   d   s   d   s   
B bit:     0   1   1   ?   0   0   1   ?   ?   ?   1   0   ?   0   0   0   ?   1   1   0   

Matching positions : [0, 1, 2, 4, 5, 6, 10, 11, 13, 14, 15, 17, 18, 19]
Alice's sifted key : [0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1]
Bob's   sifted key : [0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0]

Errors     : 4/14 = 28.6%
Threshold  : 10%
⚠ ATTACK DETECTED —